# 28 — Create And Use Custom Skills

Use this notebook as a guide for authoring a new skill, saving it into a local catalog, and attaching that skill to both `Agent` and `DeepAgent`.

In [ ]:
from pathlib import Path
import sys

ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import Markdown, display
from examples.run_multi_tool_agent import build_llm_from_env
from shipit_agent import Agent, DeepAgent, SkillCatalog, create_skill, skill_id_from_name

llm = build_llm_from_env("bedrock")

## 1. Choose A Local Catalog File

This keeps your custom skills separate from the packaged catalog.

In [ ]:
custom_catalog_path = ROOT / "notebooks" / "custom_skills.demo.json"
catalog = SkillCatalog(custom_catalog_path)
print(f"Custom catalog path: {custom_catalog_path}")

## 2. Create A New Skill

You can generate the id automatically from the name or supply it yourself.

In [ ]:
skill_name = "Release Notes Writer"
print("Generated skill id:", skill_id_from_name(skill_name))

release_notes_skill = create_skill(
    name=skill_name,
    description="Write concise product and engineering release notes from changes.",
    category="Development",
    tags=["release", "changelog", "docs"],
    use_cases=[
        "Summarize merged pull requests into a release announcement",
        "Write internal release notes for engineering and support teams",
    ],
    how_to_use=[
        "Provide merged changes, commits, or pull request summaries",
        "Ask for highlights, fixes, known issues, and upgrade notes",
    ],
    trigger_phrases=[
        "write release notes",
        "create a changelog",
        "summarize this release",
    ],
    prompt_template=(
        "You are operating with the Release Notes Writer skill. "
        "Organize the answer into Highlights, Fixes, Known Issues, and Upgrade Notes. "
        "Prefer concise bullets and call out breaking changes clearly."
    ),
)

release_notes_skill

## 3. Save The Skill Into The Catalog

The helper writes a valid `skills.json`-style file.

In [ ]:
catalog.add(release_notes_skill)
print(f"Saved {release_notes_skill.id!r} to {custom_catalog_path}")
print("Catalog now contains:")
for skill in catalog.list():
    print(f"- {skill.id}")

## 4. Create A Skill In One Call

If you do not need to hold the `Skill` object first, use `catalog.create(...)`.

In [ ]:
catalog.create(
    name="Incident Summary Writer",
    description="Turn outage notes into a concise incident summary.",
    category="Operations",
    tags=["incident", "ops", "postmortem"],
    trigger_phrases=["summarize this incident", "write an incident report"],
    prompt_template=(
        "Structure the answer as Summary, Timeline, Root Cause, Impact, and Follow-ups."
    ),
)

print("Catalog after second skill:")
for skill in catalog.list():
    print(f"- {skill.id}")

## 5. Use The Custom Skill In An Agent

Point `skill_source` at the custom catalog and attach the skill by id.

In [ ]:
agent = Agent.with_builtins(
    llm=llm,
    name="custom-skill-agent",
    skill_source=custom_catalog_path,
    auto_use_skills=False,
    skills=["release-notes-writer"],
)

print("Attached skills:")
for skill in agent.skills:
    print(f"- {skill.id}")

In [ ]:
result = agent.run(
    "Write release notes for these changes: fixed login redirect loop, improved dashboard load time, and renamed the billing settings page."
)

display(Markdown(result.output))

## 6. Use The Same Custom Skill In A DeepAgent

`DeepAgent` accepts the same `skill_source` and `skills=[...]` configuration.

In [ ]:
deep_agent = DeepAgent.with_builtins(
    llm=llm,
    name="custom-skill-deep-agent",
    skill_source=custom_catalog_path,
    auto_use_skills=False,
    skills=["release-notes-writer"],
)

print("DeepAgent attached skills:")
for skill in deep_agent.skills:
    print(f"- {skill.id}")

In [ ]:
deep_result = deep_agent.run(
    "Write release notes for these changes: fixed login redirect loop, improved dashboard load time, and renamed the billing settings page."
)

display(Markdown(getattr(deep_result, "output", str(deep_result))))

## 7. Runtime Management

You can search the local catalog and add skills after construction too.

In [ ]:
print("Local catalog search:")
for skill in agent.search_skills("incident"):
    print(f"- {skill.id}")

agent.add_skill("incident-summary-writer")
deep_agent.add_skill("incident-summary-writer")
print("\nAttached skills after add_skill:")
for skill in agent.skills:
    print(f"Agent: {skill.id}")
for skill in deep_agent.skills:
    print(f"DeepAgent: {skill.id}")

## 8. Summary

Workflow:

- create a `Skill` with `create_skill(...)`
- save it using `SkillCatalog.add(...)` or `SkillCatalog.create(...)`
- point `Agent` or `DeepAgent` at that `skill_source`
- attach the skill with `skills=[...]` or `add_skill(...)`